In [3]:
from pathlib import Path
import pandas as pd
import zipfile
import json
import numbers
from datetime import datetime

BASE = Path.home() / "BigData_UPAO" / "proyecto_afp_v2"
DRIVE = BASE / "inputs" / "drive"
EVIDENCIAS = BASE / "evidencias"

EVIDENCIAS.mkdir(parents=True, exist_ok=True)

HDFS_BASE = "hdfs://localhost:9000/user/junior/datalake_afp_v2"
DESTINO_HDFS = HDFS_BASE + "/processed/sbs_indicadores_adicionales"

print("Proyecto:", BASE)
print("Fuentes:", DRIVE)
print("Destino HDFS:", DESTINO_HDFS)
print("Existe carpeta de fuentes:", DRIVE.exists())

Proyecto: /home/junior/BigData_UPAO/proyecto_afp_v2
Fuentes: /home/junior/BigData_UPAO/proyecto_afp_v2/inputs/drive
Destino HDFS: hdfs://localhost:9000/user/junior/datalake_afp_v2/processed/sbs_indicadores_adicionales
Existe carpeta de fuentes: True


In [4]:
archivos_excel = sorted(
    p for p in DRIVE.rglob("*")
    if p.is_file() and p.suffix.lower() in (".xls", ".xlsx")
)

print("ARCHIVOS EXCEL ENCONTRADOS:", len(archivos_excel))

for ruta in archivos_excel:
    print("-", ruta.name)

ARCHIVOS EXCEL ENCONTRADOS: 3
- Afiliación y Traspasos (Desde 1993).XLS
- Inversiones y Rentabilidad (Desde 1993).XLS
- Valores cuota (Desde 1993).XLS


In [5]:
HOJAS_OBJETIVO = {
    "Mov Fondo x Traspasos-Ingresos": ("mov_fondo_ingresos", "miles_soles"),
    "Mov Fondo x Traspasos-Salidas": ("mov_fondo_salidas", "miles_soles"),
    "Mov Fondo x Traspasos-Neto": ("mov_fondo_neto", "miles_soles"),

    "Traspasos Afiliados-Ingresos": ("traspasos_afiliados_ingresos", "afiliados"),
    "Traspasos Afiliados-Salidas": ("traspasos_afiliados_salidas", "afiliados"),
    "Traspasos Afiliados-Neto": ("traspasos_afiliados_neto", "afiliados"),

    "Fondo-Cartera-Encaje": ("fondo_cartera_encaje", "millones_soles"),
    "Cartera por moneda": ("cartera_por_moneda", "miles_soles"),
}

hojas_encontradas = []

for ruta in archivos_excel:
    motor = "openpyxl" if zipfile.is_zipfile(ruta) else "xlrd"
    libro = pd.ExcelFile(ruta, engine=motor)

    for hoja_real in libro.sheet_names:
        nombre_normalizado = hoja_real.strip()

        if nombre_normalizado in HOJAS_OBJETIVO:
            codigo, unidad = HOJAS_OBJETIVO[nombre_normalizado]

            hojas_encontradas.append({
                "ruta": ruta,
                "hoja": hoja_real,
                "codigo": codigo,
                "unidad": unidad,
                "motor": motor
            })

print("HOJAS OBJETIVO ENCONTRADAS:", len(hojas_encontradas))

for item in hojas_encontradas:
    print(
        item["ruta"].name,
        "|", repr(item["hoja"]),
        "|", item["codigo"]
    )

HOJAS OBJETIVO ENCONTRADAS: 8
Afiliación y Traspasos (Desde 1993).XLS | 'Mov Fondo x Traspasos-Ingresos ' | mov_fondo_ingresos
Afiliación y Traspasos (Desde 1993).XLS | 'Mov Fondo x Traspasos-Salidas' | mov_fondo_salidas
Afiliación y Traspasos (Desde 1993).XLS | 'Mov Fondo x Traspasos-Neto' | mov_fondo_neto
Afiliación y Traspasos (Desde 1993).XLS | 'Traspasos Afiliados-Ingresos' | traspasos_afiliados_ingresos
Afiliación y Traspasos (Desde 1993).XLS | 'Traspasos Afiliados-Salidas' | traspasos_afiliados_salidas
Afiliación y Traspasos (Desde 1993).XLS | 'Traspasos Afiliados-Neto' | traspasos_afiliados_neto
Inversiones y Rentabilidad (Desde 1993).XLS | 'Fondo-Cartera-Encaje' | fondo_cartera_encaje
Inversiones y Rentabilidad (Desde 1993).XLS | 'Cartera por moneda' | cartera_por_moneda


In [6]:
import datetime as dt
import math
import re

def convertir_fecha(valor):
    if isinstance(valor, (pd.Timestamp, datetime, dt.date)):
        try:
            fecha = pd.Timestamp(valor)
            if 1990 <= fecha.year <= 2030:
                return fecha.date()
        except (ValueError, TypeError):
            return None
    return None


def convertir_numero(valor):
    if isinstance(valor, bool) or pd.isna(valor):
        return None

    if isinstance(valor, numbers.Number):
        numero = float(valor)
        return numero if math.isfinite(numero) else None

    # No interpretamos textos arbitrarios como números.
    # La auditoría SBS identificó principalmente valores numéricos nativos.
    return None


def obtener_etiqueta(datos, fila_inicial, columna):
    etiquetas = []

    for i in range(max(0, fila_inicial - 4), fila_inicial):
        valor = datos.iat[i, columna]

        if pd.notna(valor):
            texto = str(valor).strip()

            if texto and not isinstance(valor, (datetime, dt.date)):
                etiquetas.append(texto)

    return " | ".join(etiquetas[-2:]) if etiquetas else f"columna_{columna}"

In [7]:
registros = []
resumen_hojas = []

for item in hojas_encontradas:

    datos = pd.read_excel(
        item["ruta"],
        sheet_name=item["hoja"],
        engine=item["motor"],
        header=None,
        dtype=object
    )

    # Detectar la columna de fechas mediante la presencia
    # de fechas reales en las primeras filas de datos.
    candidatos = []

    for columna in datos.columns:
        fechas = datos[columna].map(convertir_fecha)
        cantidad = int(fechas.notna().sum())

        if cantidad >= 12:
            candidatos.append((columna, cantidad))

    if not candidatos:
        print("SIN COLUMNA DE FECHA:", item["hoja"])
        continue

    columna_fecha = max(candidatos, key=lambda x: x[1])[0]

    fechas = datos[columna_fecha].map(convertir_fecha)
    primera_fila = fechas.first_valid_index()

    inicio = len(registros)

    for indice_fila in range(primera_fila, len(datos)):

        fecha = fechas.iloc[indice_fila]

        if fecha is None:
            continue

        for indice_columna in range(len(datos.columns)):

            if indice_columna == columna_fecha:
                continue

            valor = convertir_numero(
                datos.iat[indice_fila, indice_columna]
            )

            if valor is None:
                continue

            registros.append({
                "fecha": fecha.isoformat(),
                "anio": fecha.year,
                "indicador": item["codigo"],
                "unidad": item["unidad"],
                "etiqueta_original": obtener_etiqueta(
                    datos, primera_fila, indice_columna
                ),
                "valor": valor,
                "archivo_origen": item["ruta"].name,
                "hoja_origen": item["hoja"].strip(),
                "fila_excel": int(indice_fila + 1),
                "columna_excel": int(indice_columna + 1)
            })

    cantidad_extraida = len(registros) - inicio

    resumen_hojas.append({
        "hoja": item["hoja"].strip(),
        "columna_fecha": int(columna_fecha + 1),
        "registros": cantidad_extraida
    })

    print(
        item["codigo"],
        "| columna fecha:", columna_fecha + 1,
        "| registros:", cantidad_extraida
    )

print("\nTOTAL DE OBSERVACIONES ADICIONALES:", len(registros))

mov_fondo_ingresos | columna fecha: 2 | registros: 1950
mov_fondo_salidas | columna fecha: 1 | registros: 2112
mov_fondo_neto | columna fecha: 2 | registros: 1611
traspasos_afiliados_ingresos | columna fecha: 1 | registros: 1950
traspasos_afiliados_salidas | columna fecha: 1 | registros: 1950
traspasos_afiliados_neto | columna fecha: 1 | registros: 1611
fondo_cartera_encaje | columna fecha: 1 | registros: 1065
cartera_por_moneda | columna fecha: 1 | registros: 4849

TOTAL DE OBSERVACIONES ADICIONALES: 17098


In [9]:
df_adicional = pd.DataFrame(registros)

print("=== RESUMEN DE REGISTROS ===")

print(
    pd.DataFrame(resumen_hojas).to_string(index=False)
)

print("\n=== MUESTRA DE OBSERVACIONES ===")

print(
    df_adicional[
        [
            "fecha",
            "indicador",
            "etiqueta_original",
            "valor",
            "hoja_origen"
        ]
    ].head(15).to_string(index=False)
)

print("\n=== CONTROL DE DUPLICADOS DE PROCEDENCIA ===")

claves = [
    "archivo_origen",
    "hoja_origen",
    "fila_excel",
    "columna_excel"
]

duplicados = df_adicional.duplicated(subset=claves).sum()

print("Duplicados por posición original:", duplicados)

print("\n=== COMPROBACIÓN DEL UMBRAL ===")

filas_actuales = 93309
filas_adicionales = len(df_adicional)

print("Filas de las siete tablas actuales:", filas_actuales)
print("Nuevas filas candidatas:", filas_adicionales)
print("Total potencial:", filas_actuales + filas_adicionales)
print(
    "Supera las 100000 filas:",
    filas_actuales + filas_adicionales > 100000
)

=== RESUMEN DE REGISTROS ===
                          hoja  columna_fecha  registros
Mov Fondo x Traspasos-Ingresos              2       1950
 Mov Fondo x Traspasos-Salidas              1       2112
    Mov Fondo x Traspasos-Neto              2       1611
  Traspasos Afiliados-Ingresos              1       1950
   Traspasos Afiliados-Salidas              1       1950
      Traspasos Afiliados-Neto              1       1611
          Fondo-Cartera-Encaje              1       1065
            Cartera por moneda              1       4849

=== MUESTRA DE OBSERVACIONES ===
     fecha          indicador etiqueta_original     valor                    hoja_origen
1996-04-01 mov_fondo_ingresos         Horizonte  1239.159 Mov Fondo x Traspasos-Ingresos
1996-04-01 mov_fondo_ingresos           Integra  2007.778 Mov Fondo x Traspasos-Ingresos
1996-04-01 mov_fondo_ingresos         Profuturo  1189.037 Mov Fondo x Traspasos-Ingresos
1996-04-01 mov_fondo_ingresos  Unión Vida 1/ 2/   515.057 Mov Fondo 

In [10]:
print("=== COMPROBACIÓN DEL NOTEBOOK ===")

print("Registros preparados:", len(registros))
print("Hojas procesadas:", len(resumen_hojas))

=== COMPROBACIÓN DEL NOTEBOOK ===
Registros preparados: 17098
Hojas procesadas: 8


In [11]:
print("=== COMPROBACIÓN DE RECUPERACIÓN ===")

print("Registros preparados:", len(registros))
print("Hojas procesadas:", len(resumen_hojas))
print("Filas del DataFrame:", len(df_adicional))

assert len(registros) == 17098
assert len(resumen_hojas) == 8
assert len(df_adicional) == 17098

print("RECUPERACIÓN CORRECTA")

=== COMPROBACIÓN DE RECUPERACIÓN ===
Registros preparados: 17098
Hojas procesadas: 8
Filas del DataFrame: 17098
RECUPERACIÓN CORRECTA


In [12]:
from pathlib import Path

CARPETA_TEMPORAL = BASE / "salidas_locales"
CARPETA_TEMPORAL.mkdir(parents=True, exist_ok=True)

ARCHIVO_RESPALDO = (
    CARPETA_TEMPORAL / "sbs_indicadores_adicionales.csv"
)

df_adicional.to_csv(
    ARCHIVO_RESPALDO,
    index=False,
    encoding="utf-8"
)

print("Respaldo guardado:", ARCHIVO_RESPALDO)
print("Registros guardados:", len(df_adicional))

Respaldo guardado: /home/junior/BigData_UPAO/proyecto_afp_v2/salidas_locales/sbs_indicadores_adicionales.csv
Registros guardados: 17098


In [8]:
import pandas as pd
import numpy as np

print("=== VALIDACIÓN ANTES DE SPARK ===")

print("Registros:", len(df_adicional))
print("Indicadores:", df_adicional["indicador"].nunique())
print("Hojas:", df_adicional["hoja_origen"].nunique())

print("\n=== NULOS EN CAMPOS OBLIGATORIOS ===")

campos = [
    "fecha",
    "indicador",
    "valor",
    "archivo_origen",
    "hoja_origen",
    "fila_excel",
    "columna_excel"
]

print(df_adicional[campos].isna().sum())

print("\n=== FECHAS ===")

fechas = pd.to_datetime(df_adicional["fecha"], errors="coerce")

print("Fecha mínima:", fechas.min())
print("Fecha máxima:", fechas.max())
print("Fechas inválidas:", fechas.isna().sum())

print("\n=== VALORES NO FINITOS ===")

print(
    "Valores infinitos:",
    (~np.isfinite(df_adicional["valor"].astype(float))).sum()
)

print("\n=== DUPLICADOS DE PROCEDENCIA ===")

claves = [
    "archivo_origen",
    "hoja_origen",
    "fila_excel",
    "columna_excel"
]

print("Duplicados:", df_adicional.duplicated(claves).sum())

print("\n=== OBSERVACIONES POR INDICADOR ===")

print(
    df_adicional.groupby("indicador")
    .agg(
        registros=("valor", "size"),
        fecha_inicio=("fecha", "min"),
        fecha_fin=("fecha", "max")
    )
    .to_string()
)

=== VALIDACIÓN ANTES DE SPARK ===
Registros: 17098
Indicadores: 8
Hojas: 8

=== NULOS EN CAMPOS OBLIGATORIOS ===
fecha             0
indicador         0
valor             0
archivo_origen    0
hoja_origen       0
fila_excel        0
columna_excel     0
dtype: int64

=== FECHAS ===
Fecha mínima: 1993-08-01 00:00:00
Fecha máxima: 2026-04-30 00:00:00
Fechas inválidas: 0

=== VALORES NO FINITOS ===
Valores infinitos: 0

=== DUPLICADOS DE PROCEDENCIA ===
Duplicados: 0

=== OBSERVACIONES POR INDICADOR ===
                              registros fecha_inicio   fecha_fin
indicador                                                       
cartera_por_moneda                 4849   1996-12-01  2026-04-30
fondo_cartera_encaje               1065   1993-08-01  2026-04-30
mov_fondo_ingresos                 1950   1996-04-01  2026-04-30
mov_fondo_neto                     1611   1996-04-01  2026-04-01
mov_fondo_salidas                  2112   1996-04-01  2026-04-30
traspasos_afiliados_ingresos       1950 

In [14]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

os.environ["HADOOP_HOME"] = "/opt/hadoop-3.3.6"
os.environ["HADOOP_CONF_DIR"] = "/opt/hadoop-3.3.6/etc/hadoop"
os.environ["YARN_CONF_DIR"] = os.environ["HADOOP_CONF_DIR"]

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "10.0.2.15"

derby_path = (BASE / "metastore" / "metastore_db").as_posix()

spark = (
    SparkSession.builder
    .appName("AFP_V2_Ampliacion_SBS")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.driver.host", "10.0.2.15")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.ui.port", "4050")
    .config("spark.executor.instances", "1")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "512m")
    .config(
        "spark.hadoop.javax.jdo.option.ConnectionURL",
        f"jdbc:derby:;databaseName={derby_path};create=true"
    )
    .config(
        "spark.sql.warehouse.dir",
        HDFS_BASE + "/catalog/warehouse"
    )
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("SPARK:", spark.version)
print("MASTER:", spark.sparkContext.master)
print("CATALOGO:", spark.conf.get("spark.sql.catalogImplementation"))

spark.sql("SHOW DATABASES").show()

SPARK: 3.5.1
MASTER: yarn
CATALOGO: hive
+-------------+
|    namespace|
+-------------+
|afp_analytics|
|      default|
+-------------+



In [15]:
from datetime import date

esquema = T.StructType([
    T.StructField("fecha", T.DateType(), False),
    T.StructField("indicador", T.StringType(), False),
    T.StructField("unidad", T.StringType(), True),
    T.StructField("etiqueta_original", T.StringType(), True),
    T.StructField("valor", T.DoubleType(), False),
    T.StructField("archivo_origen", T.StringType(), False),
    T.StructField("hoja_origen", T.StringType(), False),
    T.StructField("fila_excel", T.IntegerType(), False),
    T.StructField("columna_excel", T.IntegerType(), False),
    T.StructField("anio", T.IntegerType(), False)
])

filas_spark = [
    (
        date.fromisoformat(r["fecha"]),
        r["indicador"],
        r["unidad"],
        r["etiqueta_original"],
        float(r["valor"]),
        r["archivo_origen"],
        r["hoja_origen"],
        int(r["fila_excel"]),
        int(r["columna_excel"]),
        int(r["anio"])
    )
    for r in registros
]

sbs_nueva = spark.createDataFrame(filas_spark, schema=esquema)

print("=== DATOS PREPARADOS PARA HDFS ===")
print("Registros:", sbs_nueva.count())

sbs_nueva.printSchema()

sbs_nueva.show(10, truncate=False)

=== DATOS PREPARADOS PARA HDFS ===


Registros: 17098
root
 |-- fecha: date (nullable = false)
 |-- indicador: string (nullable = false)
 |-- unidad: string (nullable = true)
 |-- etiqueta_original: string (nullable = true)
 |-- valor: double (nullable = false)
 |-- archivo_origen: string (nullable = false)
 |-- hoja_origen: string (nullable = false)
 |-- fila_excel: integer (nullable = false)
 |-- columna_excel: integer (nullable = false)
 |-- anio: integer (nullable = false)

+----------+------------------+-----------+-----------------+--------+---------------------------------------+------------------------------+----------+-------------+----+
|fecha     |indicador         |unidad     |etiqueta_original|valor   |archivo_origen                         |hoja_origen                   |fila_excel|columna_excel|anio|
+----------+------------------+-----------+-----------------+--------+---------------------------------------+------------------------------+----------+-------------+----+
|1996-04-01|mov_fondo_ingresos|miles_s

In [16]:
print("=== GUARDANDO TABLA SBS ADICIONAL ===")

(
    sbs_nueva
    .repartition(1)
    .write
    .mode("overwrite")
    .partitionBy("anio")
    .parquet(DESTINO_HDFS)
)

print("Guardado en:", DESTINO_HDFS)

verificacion = spark.read.parquet(DESTINO_HDFS)

print("Registros en HDFS:", verificacion.count())

verificacion.groupBy("indicador").count().orderBy(
    "indicador"
).show(truncate=False)

=== GUARDANDO TABLA SBS ADICIONAL ===


Guardado en: hdfs://localhost:9000/user/junior/datalake_afp_v2/processed/sbs_indicadores_adicionales


Registros en HDFS: 17098


[Stage 12:=============================>                            (1 + 1) / 2]

+----------------------------+-----+
|indicador                   |count|
+----------------------------+-----+
|cartera_por_moneda          |4849 |
|fondo_cartera_encaje        |1065 |
|mov_fondo_ingresos          |1950 |
|mov_fondo_neto              |1611 |
|mov_fondo_salidas           |2112 |
|traspasos_afiliados_ingresos|1950 |
|traspasos_afiliados_neto    |1611 |
|traspasos_afiliados_salidas |1950 |
+----------------------------+-----+



In [17]:
print("=== REGISTRO EN HIVE ===")

spark.sql("CREATE DATABASE IF NOT EXISTS afp_analytics")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
afp_analytics.sbs_indicadores_adicionales
USING PARQUET
LOCATION '{DESTINO_HDFS}'
""")

spark.sql("""
MSCK REPAIR TABLE
afp_analytics.sbs_indicadores_adicionales
""")

spark.sql("""
SELECT COUNT(*) AS registros
FROM afp_analytics.sbs_indicadores_adicionales
""").show()

spark.sql("""
SHOW PARTITIONS
afp_analytics.sbs_indicadores_adicionales
""").show(50, truncate=False)

=== REGISTRO EN HIVE ===


+---------+
|registros|
+---------+
|    17098|
+---------+

+---------+
|partition|
+---------+
|anio=1993|
|anio=1994|
|anio=1995|
|anio=1996|
|anio=1997|
|anio=1998|
|anio=1999|
|anio=2000|
|anio=2001|
|anio=2002|
|anio=2003|
|anio=2004|
|anio=2005|
|anio=2006|
|anio=2007|
|anio=2008|
|anio=2009|
|anio=2010|
|anio=2011|
|anio=2012|
|anio=2013|
|anio=2014|
|anio=2015|
|anio=2016|
|anio=2017|
|anio=2018|
|anio=2019|
|anio=2020|
|anio=2021|
|anio=2022|
|anio=2023|
|anio=2024|
|anio=2025|
|anio=2026|
+---------+



In [18]:
print("=== CONTEO GENERAL DEL DATA LAKE ===")

tablas = [
    "bcrp_series_largas",
    "bcrp_json_rentabilidad",
    "sbs_afiliados",
    "sbs_rentabilidad",
    "sbs_valor_cuota",
    "memorias_indicadores",
    "tabla_analitica_mensual",
    "sbs_indicadores_adicionales"
]

resultados = []

for nombre in tablas:
    cantidad = spark.table(
        f"afp_analytics.{nombre}"
    ).count()

    resultados.append((nombre, cantidad))

total = sum(cantidad for _, cantidad in resultados)

for nombre, cantidad in resultados:
    print(f"{nombre}: {cantidad:,}")

print("\nTOTAL DE FILAS ALMACENADAS:", f"{total:,}")
print("SUPERA 100000:", total > 100000)

=== CONTEO GENERAL DEL DATA LAKE ===


bcrp_series_largas: 12,317
bcrp_json_rentabilidad: 699
sbs_afiliados: 1,148
sbs_rentabilidad: 2,898
sbs_valor_cuota: 76,059
memorias_indicadores: 48
tabla_analitica_mensual: 140
sbs_indicadores_adicionales: 17,098

TOTAL DE FILAS ALMACENADAS: 110,407
SUPERA 100000: True


In [19]:
import json
from datetime import datetime

reporte = {
    "fecha": datetime.now().isoformat(),
    "nombre_proyecto": "proyecto_afp_v2",
    "archivos_originales": 65,
    "valores_numericos_originales_sbs": 106841,
    "hojas_sbs_ampliadas": 8,
    "registros_sbs_adicionales": 17098,
    "tablas_parquet": 8,
    "filas_acumuladas": total,
    "supera_100000_filas_acumuladas": total > 100000,
    "duplicados_procedencia_sbs": int(duplicados)
}

ruta = BASE / "evidencias" / "14_ampliacion_sbs_validacion.json"
ruta.parent.mkdir(parents=True, exist_ok=True)

with ruta.open("w", encoding="utf-8") as f:
    json.dump(reporte, f, indent=4, ensure_ascii=False)

print("Evidencia guardada:", ruta)
print(json.dumps(reporte, indent=2, ensure_ascii=False))

Evidencia guardada: /home/junior/BigData_UPAO/proyecto_afp_v2/evidencias/14_ampliacion_sbs_validacion.json
{
  "fecha": "2026-10-10T16:03:46.702626",
  "nombre_proyecto": "proyecto_afp_v2",
  "archivos_originales": 65,
  "valores_numericos_originales_sbs": 106841,
  "hojas_sbs_ampliadas": 8,
  "registros_sbs_adicionales": 17098,
  "tablas_parquet": 8,
  "filas_acumuladas": 110407,
  "supera_100000_filas_acumuladas": true,
  "duplicados_procedencia_sbs": 0
}


In [20]:
import subprocess

BASE_HDFS = "/user/junior/datalake_afp_v2"

def listar_directorios(ruta):
    resultado = subprocess.run(
        ["hdfs", "dfs", "-ls", ruta],
        capture_output=True,
        text=True,
        check=True
    )

    directorios = []

    for linea in resultado.stdout.splitlines():
        partes = linea.split()

        if len(partes) >= 8 and partes[0].startswith("d"):
            directorios.append(partes[-1])

    return sorted(directorios)


def mostrar_arbol(ruta, prefijo="", nivel=0, max_nivel=3):
    if nivel >= max_nivel:
        return

    directorios = listar_directorios(ruta)

    for i, directorio in enumerate(directorios):
        ultimo = i == len(directorios) - 1
        conector = "└── " if ultimo else "├── "

        nombre = directorio.rstrip("/").split("/")[-1]

        print(prefijo + conector + nombre + "/")

        nuevo_prefijo = prefijo + (
            "    " if ultimo else "│   "
        )

        mostrar_arbol(
            directorio,
            nuevo_prefijo,
            nivel + 1,
            max_nivel
        )


print("=== ESTRUCTURA REAL DEL DATA LAKE EN HDFS ===")
print("datalake_afp_v2/")

mostrar_arbol(BASE_HDFS, max_nivel=3)

=== ESTRUCTURA REAL DEL DATA LAKE EN HDFS ===
datalake_afp_v2/
├── catalog/
│   └── warehouse/
│       └── afp_analytics.db/
├── processed/
│   ├── bcrp_json_rentabilidad/
│   │   ├── anio=2015/
│   │   ├── anio=2016/
│   │   ├── anio=2017/
│   │   ├── anio=2018/
│   │   ├── anio=2019/
│   │   ├── anio=2020/
│   │   ├── anio=2021/
│   │   ├── anio=2022/
│   │   ├── anio=2023/
│   │   ├── anio=2024/
│   │   ├── anio=2025/
│   │   └── anio=2026/
│   ├── bcrp_series_largas/
│   │   ├── anio=1993/
│   │   ├── anio=1994/
│   │   ├── anio=1995/
│   │   ├── anio=1996/
│   │   ├── anio=1997/
│   │   ├── anio=1998/
│   │   ├── anio=1999/
│   │   ├── anio=2000/
│   │   ├── anio=2001/
│   │   ├── anio=2002/
│   │   ├── anio=2003/
│   │   ├── anio=2004/
│   │   ├── anio=2005/
│   │   ├── anio=2006/
│   │   ├── anio=2007/
│   │   ├── anio=2008/
│   │   ├── anio=2009/
│   │   ├── anio=2010/
│   │   ├── anio=2011/
│   │   ├── anio=2012/
│   │   ├── anio=2013/
│   │   ├── anio=2014/
│   │   ├── anio=2

In [21]:
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Catálogo:", spark.conf.get("spark.sql.catalogImplementation"))

spark.sql("SHOW DATABASES").show()
spark.sql("SHOW TABLES IN afp_analytics").show(truncate=False)

Spark: 3.5.1
Master: yarn
Catálogo: hive
+-------------+
|    namespace|
+-------------+
|afp_analytics|
|      default|
+-------------+

+-------------+---------------------------+-----------+
|namespace    |tableName                  |isTemporary|
+-------------+---------------------------+-----------+
|afp_analytics|bcrp_json_rentabilidad     |false      |
|afp_analytics|bcrp_series_largas         |false      |
|afp_analytics|memorias_indicadores       |false      |
|afp_analytics|sbs_afiliados              |false      |
|afp_analytics|sbs_indicadores_adicionales|false      |
|afp_analytics|sbs_rentabilidad           |false      |
|afp_analytics|sbs_valor_cuota            |false      |
|afp_analytics|tabla_analitica_mensual    |false      |
+-------------+---------------------------+-----------+



In [23]:
import subprocess

BASE_HDFS = "/user/junior/datalake_afp_v2"

def carpetas_hdfs(ruta):
    salida = subprocess.run(
        ["hdfs", "dfs", "-ls", ruta],
        capture_output=True,
        text=True,
        check=True
    ).stdout

    return sorted(
        linea.split()[-1]
        for linea in salida.splitlines()
        if linea.split() and linea.split()[0].startswith("d")
    )

def arbol_hdfs(ruta, prefijo="", profundidad=0, maximo=2):
    if profundidad >= maximo:
        return

    carpetas = carpetas_hdfs(ruta)

    for i, carpeta in enumerate(carpetas):
        ultima = i == len(carpetas) - 1
        nombre = carpeta.rstrip("/").split("/")[-1]

        print(prefijo + ("└── " if ultima else "├── ") + nombre + "/")

        arbol_hdfs(
            carpeta,
            prefijo + ("    " if ultima else "│   "),
            profundidad + 1,
            maximo
        )

print("datalake_afp_v2/")
arbol_hdfs(BASE_HDFS)

datalake_afp_v2/
├── catalog/
│   └── warehouse/
├── processed/
│   ├── bcrp_json_rentabilidad/
│   ├── bcrp_series_largas/
│   ├── memorias_indicadores/
│   ├── sbs_afiliados/
│   ├── sbs_indicadores_adicionales/
│   ├── sbs_rentabilidad/
│   ├── sbs_valor_cuota/
│   └── tabla_analitica_mensual/
├── pruebas/
├── quality/
│   └── uniones_sin_correspondencia/
├── raw/
│   ├── _manifests/
│   ├── afp_memorias/
│   ├── bcrp/
│   ├── control/
│   └── sbs/
└── staging/
    ├── documentos/
    └── sbs/


In [25]:
tabla = spark.table(
    "afp_analytics.tabla_analitica_mensual"
)

tabla.select(
    "fecha_mes",
    "prom_otras3",
    "brecha_real_pp",
    "brecha_lag1",
    "brecha_lag3",
    "sistema_lag1",
    "ipc_lag1",
    "tipo_cambio_lag1",
    "objetivo_bajo_rendimiento_mes_siguiente"
).orderBy("fecha_mes").show(10, truncate=False)

+----------+-------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+---------------------------------------+
|fecha_mes |prom_otras3        |brecha_real_pp      |brecha_lag1         |brecha_lag3         |sistema_lag1|ipc_lag1          |tipo_cambio_lag1|objetivo_bajo_rendimiento_mes_siguiente|
+----------+-------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+---------------------------------------+
|2015-01-01|5.131766666666667  |-0.1480666666666668 |NULL                |NULL                |NULL        |0.228891578078382 |2.9615          |1                                      |
|2015-02-01|9.8595             |-0.28630000000000067|-0.1480666666666668 |NULL                |4.9809      |0.170295685735701 |3.005525        |1                                      |
|2015-03-01|8.508833333333333  |-0.45743333333333425|-0.28630000000000067|N

In [26]:
from pyspark.sql import functions as F

final = spark.table(
    "afp_analytics.tabla_analitica_mensual"
)

final.agg(
    F.count("*").alias("filas"),
    F.countDistinct("fecha_mes").alias("meses_unicos"),
    F.min("fecha_mes").alias("fecha_inicio"),
    F.max("fecha_mes").alias("fecha_fin")
).show(truncate=False)

print(
    "Meses duplicados:",
    final.groupBy("fecha_mes")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

final.groupBy("fuente_sistema").count().show(
    truncate=False
)

+-----+------------+------------+----------+
|filas|meses_unicos|fecha_inicio|fecha_fin |
+-----+------------+------------+----------+
|140  |140         |2015-01-01  |2026-08-01|
+-----+------------+------------+----------+

Meses duplicados: 0
+--------------------------+-----+
|fuente_sistema            |count|
+--------------------------+-----+
|BCRP_JSON                 |139  |
|SBS_XLS_CORRECCION_2018_12|1    |
+--------------------------+-----+

